# Modelado V5 con selección automática de mejor modelo por producto

Este notebook reemplaza el enfoque de entrenar únicamente `RandomForestRegressor`.

Ahora, para cada producto, prueba varios modelos, compara sus métricas en validación cronológica, selecciona el mejor y luego entrena el modelo final con entrenamiento + validación.

El conjunto de prueba se usa solo para medir el desempeño final, no para escoger el modelo.

## Bloque 1. Montar Drive e importar librerías

Este bloque conecta Google Drive, importa librerías y carga los modelos de scikit-learn que se compararán.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import joblib
import json
from pathlib import Path

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Bloque 2. Rutas y configuración

Ajusta estas rutas según tu entorno.

En tu caso actual, `dataset_preparado_v3.csv` está temporalmente en `/content` y la carpeta `datasets_por_plato` está en `MyDrive`.

In [8]:
# ============================================================
# 1. RUTAS Y CONFIGURACIÓN
# ============================================================

# Dataset preparado en CSV.
# Si lo mueves a Drive, cambia esta ruta.
INPUT_PREPARADO = "/content/dataset_preparado_v3.csv"

# Carpeta donde están las subcarpetas:
# almuerzo, sopa, fanesca, colada_morada
RUTA_DIVISION = "/content/drive/MyDrive/datasets_por_plato"

# Carpeta donde se guardarán los mejores modelos seleccionados.
# Se usa un nombre genérico porque ya no necesariamente todos serán Random Forest.
OUTPUT_MODELOS = "/content/drive/MyDrive/modelos_mejor_modelo"

# Reportes de salida.
OUTPUT_RESULTADOS = "/content/drive/MyDrive/resultados_modelado_mejor_modelo.xlsx"
OUTPUT_CONFIG_PKL = f"{OUTPUT_MODELOS}/config_entrenamiento.pkl"
OUTPUT_CONFIG_JSON = f"{OUTPUT_MODELOS}/config_entrenamiento.json"

os.makedirs(OUTPUT_MODELOS, exist_ok=True)

PRODUCTOS = ["almuerzo", "sopa", "fanesca", "colada_morada"]

VARIABLES_PREDICTORAS = [
    "anio",
    "mes",
    "dia_mes",
    "dia_semana_num",
    "semana_anio",

    "es_fanesca_temporada",
    "es_colada_temporada",
    "es_inicio_mes",
    "es_quincena",
    "es_fin_mes",

    "es_lunes",
    "es_martes",
    "es_miercoles",
    "es_jueves",
    "es_viernes",

    "mes_sin",
    "mes_cos",
    "dia_semana_sin",
    "dia_semana_cos",

    "tendencia",
    "tendencia_log",
    "crecimiento_anual",

    "preciomenu",
    "preciosopa",
    "fanesca_precio",
    "coladamorada_precio"
]

# Criterio principal de selección.
# Se selecciona el modelo con menor RMSE en validación.
# Si hay empate, se usa menor MAE y luego mayor R².
CRITERIO_PRINCIPAL = "RMSE"

## Bloque 3. Modelos candidatos

Aquí se definen los modelos que se probarán por producto.

Se incluyen modelos simples, modelos lineales regularizados y modelos de árboles.

También se incluye un modelo base (`Baseline_Promedio`) para saber si los modelos realmente mejoran frente a una predicción promedio.

In [9]:
# ============================================================
# 2. MODELOS CANDIDATOS
# ============================================================

MODELOS_CANDIDATOS = {
    "Baseline_Promedio": DummyRegressor(strategy="mean"),

    "Regresion_Lineal": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0, random_state=42))
    ]),

    "ElasticNet": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.05, l1_ratio=0.2, random_state=42, max_iter=10000))
    ]),

    "RandomForest": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        max_depth=6,
        min_samples_split=8,
        min_samples_leaf=4,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        max_depth=8,
        min_samples_split=6,
        min_samples_leaf=3,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=4
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42,
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1
    )
}

print("Modelos candidatos:")
for nombre in MODELOS_CANDIDATOS:
    print("-", nombre)

Modelos candidatos:
- Baseline_Promedio
- Regresion_Lineal
- Ridge
- ElasticNet
- RandomForest
- ExtraTrees
- GradientBoosting
- HistGradientBoosting


## Bloque 4. Lectura robusta del dataset preparado y configuración base

Este bloque lee el dataset preparado, soportando CSV con separador `;` o `,`.

También valida columnas y rango de fechas.

In [10]:
# ============================================================
# 3. LECTURA ROBUSTA DEL DATASET PREPARADO
# ============================================================

def leer_dataset_preparado(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        print("Leyendo archivo Excel...")
        return pd.read_excel(path)

    if path.suffix.lower() == ".csv":
        print("Leyendo archivo CSV...")

        intentos = [
            {"sep": ";", "encoding": "utf-8-sig"},
            {"sep": ",", "encoding": "utf-8-sig"},
            {"sep": ";", "encoding": "latin-1"},
            {"sep": ",", "encoding": "latin-1"},
            {"sep": None, "encoding": "utf-8-sig"},
            {"sep": None, "encoding": "latin-1"},
        ]

        ultimo_error = None

        for intento in intentos:
            try:
                if intento["sep"] is None:
                    df_temp = pd.read_csv(
                        path,
                        sep=None,
                        engine="python",
                        encoding=intento["encoding"]
                    )
                else:
                    df_temp = pd.read_csv(
                        path,
                        sep=intento["sep"],
                        engine="python",
                        encoding=intento["encoding"]
                    )

                if df_temp.shape[1] > 1:
                    print(
                        "Archivo leído correctamente con:",
                        f"sep={repr(intento['sep'])},",
                        f"encoding={intento['encoding']}"
                    )
                    return df_temp

            except Exception as e:
                ultimo_error = e

        raise ValueError(
            f"No se pudo leer el CSV correctamente. Último error: {ultimo_error}"
        )

    raise ValueError("Formato no soportado. Usa .csv, .xlsx o .xls.")


df_base = leer_dataset_preparado(INPUT_PREPARADO)

df_base.columns = [str(c).strip() for c in df_base.columns]

if "fecha" not in df_base.columns:
    raise ValueError("El dataset preparado no contiene la columna 'fecha'.")

df_base["fecha"] = pd.to_datetime(df_base["fecha"], errors="coerce")
df_base = df_base.dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

faltantes = [
    c for c in ["fecha"] + VARIABLES_PREDICTORAS + PRODUCTOS
    if c not in df_base.columns
]

if faltantes:
    raise ValueError(f"Faltan columnas en dataset_preparado: {faltantes}")

print("Dataset preparado leído correctamente.")
print("Filas:", len(df_base))
print("Rango:", df_base["fecha"].min().date(), "a", df_base["fecha"].max().date())

display(df_base.head())

Leyendo archivo CSV...
Archivo leído correctamente con: sep=';', encoding=utf-8-sig
Dataset preparado leído correctamente.
Filas: 740
Rango: 2023-01-02 a 2025-12-31


,fecha,anio,mes,dia_mes,dia_semana_num,semana_anio,es_fanesca_temporada,es_colada_temporada,es_inicio_mes,es_quincena,...,tendencia_log,crecimiento_anual,preciomenu,preciosopa,fanesca_precio,coladamorada_precio,almuerzo,sopa,fanesca,colada_morada
0,2023-01-02,2023,1,2,0,1,0,0,1,0,...,0,0,4,"1,5",7,2,104,72,0,0
1,2023-01-03,2023,1,3,1,1,0,0,1,0,...,"0,693147181",0,4,"1,5",7,2,106,54,0,0
2,2023-01-04,2023,1,4,2,1,0,0,1,0,...,"1,098612289",0,4,"1,5",7,2,97,72,0,0
3,2023-01-05,2023,1,5,3,1,0,0,1,0,...,"1,386294361",0,4,"1,5",7,2,103,0,0,0
4,2023-01-06,2023,1,6,4,1,0,0,0,0,...,"1,609437912",0,4,"1,5",7,2,123,0,0,0


## Bloque 5. Funciones de preparación, métricas e interpretación

Estas funciones se usan durante el entrenamiento:

- Limpian números con coma decimal.
- Preparan train, validación y prueba.
- Calculan métricas.
- Extraen importancia de variables cuando el modelo lo permite.
- Generan textos explicativos para el reporte.

In [11]:
# ============================================================
# 4. FUNCIONES AUXILIARES
# ============================================================

def convertir_numero_seguro(serie):
    # Convierte '0,5' a 0.5 y maneja nulos.
    return (
        serie
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace(["", "nan", "None", "NaN", "NULL"], np.nan)
        .pipe(pd.to_numeric, errors="coerce")
    )


def preparar_conjunto_modelo(df, nombre_conjunto, producto):
    df = df.copy()

    columnas_necesarias = ["fecha"] + VARIABLES_PREDICTORAS + ["demanda"]

    faltantes = [c for c in columnas_necesarias if c not in df.columns]

    if faltantes:
        raise ValueError(f"Faltan columnas en {nombre_conjunto}_{producto}: {faltantes}")

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    for col in VARIABLES_PREDICTORAS:
        df[col] = convertir_numero_seguro(df[col])

    df["demanda"] = convertir_numero_seguro(df["demanda"])

    columnas_con_nulos = df[VARIABLES_PREDICTORAS + ["demanda"]].columns[
        df[VARIABLES_PREDICTORAS + ["demanda"]].isna().any()
    ].tolist()

    if columnas_con_nulos:
        print(f"Advertencia en {nombre_conjunto}_{producto}: columnas con nulos convertidos a 0:")
        print(columnas_con_nulos)

    df[VARIABLES_PREDICTORAS] = df[VARIABLES_PREDICTORAS].fillna(0)
    df["demanda"] = df["demanda"].fillna(0)

    return df


def calcular_metricas(y_real, y_pred):
    y_real = np.asarray(y_real, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_real, y_pred)
    rmse = mean_squared_error(y_real, y_pred) ** 0.5

    try:
        r2 = r2_score(y_real, y_pred)
    except Exception:
        r2 = np.nan

    if np.sum(y_real) != 0:
        wape = np.sum(np.abs(y_real - y_pred)) / np.sum(y_real) * 100
    else:
        wape = np.nan

    mask_mape = y_real != 0
    if np.any(mask_mape):
        mape = np.mean(np.abs((y_real[mask_mape] - y_pred[mask_mape]) / y_real[mask_mape])) * 100
    else:
        mape = np.nan

    sesgo = np.mean(y_pred - y_real)

    return {
        "MAE": round(float(mae), 3),
        "RMSE": round(float(rmse), 3),
        "R2": round(float(r2), 4) if pd.notna(r2) else np.nan,
        "WAPE_%": round(float(wape), 3) if pd.notna(wape) else np.nan,
        "MAPE_%": round(float(mape), 3) if pd.notna(mape) else np.nan,
        "Sesgo_promedio": round(float(sesgo), 3)
    }


def obtener_estimador_final(modelo):
    if isinstance(modelo, Pipeline):
        return modelo.steps[-1][1]
    return modelo


def extraer_importancias(modelo, producto):
    estimador = obtener_estimador_final(modelo)

    if hasattr(estimador, "feature_importances_"):
        return pd.DataFrame({
            "producto": producto,
            "tipo_importancia": "feature_importances",
            "variable": VARIABLES_PREDICTORAS,
            "importancia": estimador.feature_importances_
        }).sort_values("importancia", ascending=False)

    if hasattr(estimador, "coef_"):
        coef = np.ravel(estimador.coef_)
        if len(coef) == len(VARIABLES_PREDICTORAS):
            return pd.DataFrame({
                "producto": producto,
                "tipo_importancia": "coeficiente_absoluto",
                "variable": VARIABLES_PREDICTORAS,
                "importancia": np.abs(coef),
                "coeficiente": coef
            }).sort_values("importancia", ascending=False)

    return pd.DataFrame()


def explicar_seleccion(producto, mejor_fila):
    return (
        f"Para el producto {producto}, el modelo seleccionado fue {mejor_fila['modelo_candidato']}. "
        f"La selección se realizó usando validación cronológica y el criterio principal fue menor RMSE. "
        f"Este criterio es adecuado porque el RMSE penaliza con mayor fuerza los errores grandes, "
        f"lo cual es importante en predicción de demanda porque un pico mal estimado puede afectar planificación, compras e inventario. "
        f"El modelo seleccionado obtuvo en validación: "
        f"MAE={mejor_fila['MAE']}, RMSE={mejor_fila['RMSE']}, R2={mejor_fila['R2']}, "
        f"WAPE={mejor_fila['WAPE_%']}% y MAPE={mejor_fila['MAPE_%']}%. "
        f"MAE se incluye para interpretar el error promedio en unidades, RMSE para priorizar errores grandes "
        f"y R2 para revisar capacidad explicativa del modelo."
    )

## Bloque 6. Entrenar modelos candidatos y seleccionar el mejor por producto

Para cada producto:

1. Lee train, validación y prueba.
2. Entrena todos los modelos candidatos usando train.
3. Evalúa en validación.
4. Selecciona el mejor con menor RMSE.
5. Reentrena el modelo seleccionado con train + validación.
6. Evalúa el modelo final en prueba.
7. Guarda el `.pkl` final del producto.

In [12]:
# ============================================================
# 5. ENTRENAMIENTO, COMPARACIÓN Y SELECCIÓN POR PRODUCTO
# ============================================================

comparacion_validacion = []
metricas_finales = []
predicciones_finales = []
importancias_todas = []
seleccion_modelos = []
reporte_texto = []
metricas_estacionales = []

modelo_por_producto = {}
razon_por_producto = {}

for producto in PRODUCTOS:

    print(f"\n{'=' * 80}")
    print(f"Procesando producto: {producto}")
    print(f"{'=' * 80}")

    path_train = f"{RUTA_DIVISION}/{producto}/train_{producto}.xlsx"
    path_val = f"{RUTA_DIVISION}/{producto}/validacion_{producto}.xlsx"
    path_test = f"{RUTA_DIVISION}/{producto}/prueba_{producto}.xlsx"

    if not os.path.exists(path_train):
        raise FileNotFoundError(f"No existe archivo train: {path_train}")
    if not os.path.exists(path_val):
        raise FileNotFoundError(f"No existe archivo validación: {path_val}")
    if not os.path.exists(path_test):
        raise FileNotFoundError(f"No existe archivo prueba: {path_test}")

    train = pd.read_excel(path_train)
    val = pd.read_excel(path_val)
    test = pd.read_excel(path_test)

    train = preparar_conjunto_modelo(train, "train", producto)
    val = preparar_conjunto_modelo(val, "validacion", producto)
    test = preparar_conjunto_modelo(test, "prueba", producto)

    X_train = train[VARIABLES_PREDICTORAS]
    y_train = train["demanda"]

    X_val = val[VARIABLES_PREDICTORAS]
    y_val = val["demanda"]

    X_test = test[VARIABLES_PREDICTORAS]
    y_test = test["demanda"]

    # ------------------------------------------------------------
    # 1. Comparación de modelos candidatos en validación
    # ------------------------------------------------------------

    resultados_producto = []

    for nombre_modelo, modelo_base in MODELOS_CANDIDATOS.items():

        print(f"Entrenando candidato: {nombre_modelo}")

        modelo_candidato = clone(modelo_base)
        modelo_candidato.fit(X_train, y_train)

        pred_val = modelo_candidato.predict(X_val)
        pred_val = np.maximum(pred_val, 0)

        met_val = calcular_metricas(y_val, pred_val)

        fila = {
            "producto": producto,
            "modelo_candidato": nombre_modelo,
            "conjunto": "Validación",
            "registros": len(val),
            "total_real": round(float(np.sum(y_val)), 3),
            "total_predicho": round(float(np.sum(pred_val)), 3),
            **met_val
        }

        resultados_producto.append(fila)
        comparacion_validacion.append(fila)

    df_resultados_producto = pd.DataFrame(resultados_producto)

    # Selección:
    # menor RMSE, luego menor MAE, luego mayor R2.
    df_resultados_producto = df_resultados_producto.sort_values(
        by=["RMSE", "MAE", "R2"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    mejor_fila = df_resultados_producto.iloc[0]
    mejor_nombre = mejor_fila["modelo_candidato"]

    print(f"Mejor modelo para {producto}: {mejor_nombre}")

    # ------------------------------------------------------------
    # 2. Reentrenamiento final con train + validación
    # ------------------------------------------------------------

    train_val = pd.concat([train, val], ignore_index=True)
    X_train_val = train_val[VARIABLES_PREDICTORAS]
    y_train_val = train_val["demanda"]

    modelo_final = clone(MODELOS_CANDIDATOS[mejor_nombre])
    modelo_final.fit(X_train_val, y_train_val)

    # ------------------------------------------------------------
    # 3. Evaluación final en prueba
    # ------------------------------------------------------------

    pred_test = modelo_final.predict(X_test)
    pred_test = np.maximum(pred_test, 0)

    met_test = calcular_metricas(y_test, pred_test)

    fila_final = {
        "producto": producto,
        "modelo_final": mejor_nombre,
        "conjunto": "Prueba",
        "registros": len(test),
        "total_real": round(float(np.sum(y_test)), 3),
        "total_predicho": round(float(np.sum(pred_test)), 3),
        **met_test
    }

    metricas_finales.append(fila_final)

    # ------------------------------------------------------------
    # 4. Guardar predicciones finales
    # ------------------------------------------------------------

    pred_temp = test[["fecha"]].copy()
    pred_temp["producto"] = producto
    pred_temp["modelo_final"] = mejor_nombre
    pred_temp["conjunto"] = "Prueba"
    pred_temp["demanda_real"] = y_test.values
    pred_temp["demanda_predicha"] = pred_test.round().astype(int)
    pred_temp["error"] = pred_temp["demanda_predicha"] - pred_temp["demanda_real"]
    pred_temp["error_abs"] = pred_temp["error"].abs()

    predicciones_finales.append(pred_temp)

    # ------------------------------------------------------------
    # 5. Guardar importancia de variables
    # ------------------------------------------------------------

    imp = extraer_importancias(modelo_final, producto)
    if not imp.empty:
        imp["modelo_final"] = mejor_nombre
        importancias_todas.append(imp)

    # ------------------------------------------------------------
    # 6. Guardar modelo final
    # ------------------------------------------------------------

    ruta_modelo = f"{OUTPUT_MODELOS}/modelo_{producto}.pkl"
    joblib.dump(modelo_final, ruta_modelo)

    print("Modelo final guardado:", ruta_modelo)

    # ------------------------------------------------------------
    # 7. Reporte de selección
    # ------------------------------------------------------------

    explicacion = explicar_seleccion(producto, mejor_fila)

    seleccion_modelos.append({
        "producto": producto,
        "modelo_seleccionado": mejor_nombre,
        "criterio_principal": "Menor RMSE en validación",
        "criterios_secundarios": "Menor MAE y mayor R2 en caso de empate",
        "MAE_validacion": mejor_fila["MAE"],
        "RMSE_validacion": mejor_fila["RMSE"],
        "R2_validacion": mejor_fila["R2"],
        "WAPE_validacion_%": mejor_fila["WAPE_%"],
        "MAPE_validacion_%": mejor_fila["MAPE_%"],
        "MAE_prueba": met_test["MAE"],
        "RMSE_prueba": met_test["RMSE"],
        "R2_prueba": met_test["R2"],
        "WAPE_prueba_%": met_test["WAPE_%"],
        "MAPE_prueba_%": met_test["MAPE_%"],
        "ruta_modelo": ruta_modelo,
        "explicacion": explicacion
    })

    reporte_texto.append({
        "seccion": f"Selección de modelo - {producto}",
        "contenido": explicacion
    })

    modelo_por_producto[producto] = mejor_nombre
    razon_por_producto[producto] = explicacion

    # ------------------------------------------------------------
    # 8. Evaluación estacional especial si existe
    # ------------------------------------------------------------

    path_train_est = f"{RUTA_DIVISION}/{producto}/train_estacional_{producto}.xlsx"
    path_test_est = f"{RUTA_DIVISION}/{producto}/prueba_estacional_{producto}.xlsx"

    if os.path.exists(path_train_est) and os.path.exists(path_test_est):

        print(f"Evaluación estacional detectada para: {producto}")

        train_est = pd.read_excel(path_train_est)
        test_est = pd.read_excel(path_test_est)

        train_est = preparar_conjunto_modelo(train_est, "train_estacional", producto)
        test_est = preparar_conjunto_modelo(test_est, "prueba_estacional", producto)

        if len(train_est) > 0 and len(test_est) > 0:
            modelo_est = clone(MODELOS_CANDIDATOS[mejor_nombre])
            modelo_est.fit(train_est[VARIABLES_PREDICTORAS], train_est["demanda"])

            y_est = test_est["demanda"]
            pred_est = np.maximum(
                modelo_est.predict(test_est[VARIABLES_PREDICTORAS]),
                0
            )

            met_est = calcular_metricas(y_est, pred_est)

            metricas_estacionales.append({
                "producto": producto,
                "modelo_usado": mejor_nombre,
                "conjunto": "Prueba estacional",
                "registros": len(test_est),
                "total_real": round(float(np.sum(y_est)), 3),
                "total_predicho": round(float(np.sum(pred_est)), 3),
                **met_est
            })

print("\nEntrenamiento y selección automática finalizados.")


Procesando producto: almuerzo
Entrenando candidato: Baseline_Promedio
Entrenando candidato: Regresion_Lineal
Entrenando candidato: Ridge
Entrenando candidato: ElasticNet
Entrenando candidato: RandomForest
Entrenando candidato: ExtraTrees
Entrenando candidato: GradientBoosting
Entrenando candidato: HistGradientBoosting
Mejor modelo para almuerzo: HistGradientBoosting
Modelo final guardado: /content/drive/MyDrive/modelos_mejor_modelo/modelo_almuerzo.pkl

Procesando producto: sopa
Entrenando candidato: Baseline_Promedio
Entrenando candidato: Regresion_Lineal
Entrenando candidato: Ridge
Entrenando candidato: ElasticNet
Entrenando candidato: RandomForest
Entrenando candidato: ExtraTrees
Entrenando candidato: GradientBoosting
Entrenando candidato: HistGradientBoosting
Mejor modelo para sopa: ElasticNet
Modelo final guardado: /content/drive/MyDrive/modelos_mejor_modelo/modelo_sopa.pkl

Procesando producto: fanesca
Entrenando candidato: Baseline_Promedio
Entrenando candidato: Regresion_Lineal

## Bloque 7. Crear configuración final

La configuración ahora guarda qué modelo fue seleccionado para cada producto.

Esto es importante porque ya no se asume que todos los productos usan Random Forest.

In [13]:
# ============================================================
# 6. CONFIGURACIÓN FINAL
# ============================================================

config_entrenamiento = {
    "version": "Version_5_seleccion_automatica",
    "modelo": "Seleccion_automatica_por_producto",
    "enfoque": "un_modelo_por_producto_con_seleccion_automatica",
    "productos": PRODUCTOS,
    "variables_predictoras": VARIABLES_PREDICTORAS,
    "fecha_min_modelo": str(df_base["fecha"].min().date()),
    "fecha_max_modelo": str(df_base["fecha"].max().date()),
    "temporada_colada_morada": {
        "inicio_mes": 10,
        "inicio_dia": 1,
        "fin_mes": 11,
        "fin_dia": 4
    },
    "temporada_fanesca": {
        "meses": [2, 3]
    },
    "base_ciclo_dia_semana": 5,
    "criterio_seleccion": {
        "principal": "Menor RMSE en validación",
        "secundarios": ["Menor MAE", "Mayor R2"],
        "justificacion": (
            "La selección se hace con validación cronológica para evitar fuga de información. "
            "RMSE se usa como criterio principal porque penaliza más los errores grandes, "
            "lo cual es relevante en demanda operativa. MAE ayuda a interpretar el error promedio "
            "en unidades y R2 muestra capacidad explicativa."
        )
    },
    "modelos_candidatos": list(MODELOS_CANDIDATOS.keys()),
    "modelo_por_producto": modelo_por_producto,
    "razon_por_producto": razon_por_producto,
}

joblib.dump(config_entrenamiento, OUTPUT_CONFIG_PKL)

with open(OUTPUT_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(config_entrenamiento, f, ensure_ascii=False, indent=4)

print("Configuración final guardada:")
print(OUTPUT_CONFIG_PKL)
print(OUTPUT_CONFIG_JSON)

display(pd.DataFrame([
    {"producto": p, "modelo_seleccionado": m}
    for p, m in modelo_por_producto.items()
]))

Configuración final guardada:
/content/drive/MyDrive/modelos_mejor_modelo/config_entrenamiento.pkl
/content/drive/MyDrive/modelos_mejor_modelo/config_entrenamiento.json


,producto,modelo_seleccionado
0,almuerzo,HistGradientBoosting
1,sopa,ElasticNet
2,fanesca,HistGradientBoosting
3,colada_morada,RandomForest


## Bloque 8. Consolidar resultados y generar reporte

Este bloque crea tablas finales:

- Comparación de todos los modelos candidatos.
- Modelo seleccionado por producto.
- Métricas finales en prueba.
- Predicciones finales.
- Importancia de variables.
- Evaluación estacional, si aplica.
- Reporte textual con explicación del porqué se eligió cada modelo.

In [14]:
# ============================================================
# 7. CONSOLIDAR Y EXPORTAR RESULTADOS
# ============================================================

df_comparacion_validacion = pd.DataFrame(comparacion_validacion)
df_metricas_finales = pd.DataFrame(metricas_finales)
df_predicciones_finales = pd.concat(predicciones_finales, ignore_index=True) if predicciones_finales else pd.DataFrame()
df_importancias = pd.concat(importancias_todas, ignore_index=True) if importancias_todas else pd.DataFrame()
df_seleccion_modelos = pd.DataFrame(seleccion_modelos)
df_reporte_texto = pd.DataFrame(reporte_texto)
df_metricas_estacionales = pd.DataFrame(metricas_estacionales)

# Ordenar comparación por producto y RMSE.
if not df_comparacion_validacion.empty:
    df_comparacion_validacion = df_comparacion_validacion.sort_values(
        ["producto", "RMSE", "MAE", "R2"],
        ascending=[True, True, True, False]
    ).reset_index(drop=True)

with pd.ExcelWriter(OUTPUT_RESULTADOS, engine="openpyxl") as writer:
    df_seleccion_modelos.to_excel(writer, sheet_name="seleccion_modelos", index=False)
    df_comparacion_validacion.to_excel(writer, sheet_name="comparacion_validacion", index=False)
    df_metricas_finales.to_excel(writer, sheet_name="metricas_prueba", index=False)
    df_predicciones_finales.to_excel(writer, sheet_name="predicciones_prueba", index=False)

    if not df_importancias.empty:
        df_importancias.to_excel(writer, sheet_name="importancia_variables", index=False)

    if not df_metricas_estacionales.empty:
        df_metricas_estacionales.to_excel(writer, sheet_name="metricas_estacionales", index=False)

    df_reporte_texto.to_excel(writer, sheet_name="reporte_texto", index=False)

    metodologia = pd.DataFrame([
        {
            "tema": "Objetivo",
            "descripcion": "Comparar varios modelos por producto y seleccionar automáticamente el mejor según validación cronológica."
        },
        {
            "tema": "Criterio principal",
            "descripcion": "Menor RMSE en validación."
        },
        {
            "tema": "Por qué RMSE",
            "descripcion": "RMSE penaliza más los errores grandes, lo cual es importante cuando un error alto puede afectar planificación de producción, compras o inventario."
        },
        {
            "tema": "Métrica complementaria MAE",
            "descripcion": "MAE muestra el error promedio en unidades de platos, por lo que es fácil de interpretar operativamente."
        },
        {
            "tema": "Métrica complementaria R2",
            "descripcion": "R2 muestra qué tanto el modelo explica la variabilidad de los datos reales. Valores cercanos a 1 son mejores; valores negativos indican bajo desempeño."
        },
        {
            "tema": "Uso de prueba",
            "descripcion": "El conjunto de prueba no se usa para escoger el modelo. Se usa solo para medir el desempeño final del modelo seleccionado."
        },
        {
            "tema": "Productos estacionales",
            "descripcion": "Cuando existen archivos estacionales, se calcula una evaluación adicional para fanesca y colada morada en su temporada."
        }
    ])
    metodologia.to_excel(writer, sheet_name="metodologia", index=False)

print("\nModelado con selección automática finalizado correctamente.")
print("Reporte:", OUTPUT_RESULTADOS)
print("Modelos:", OUTPUT_MODELOS)
print("Configuración:", OUTPUT_CONFIG_PKL)

print("\nModelos seleccionados por producto:")
display(df_seleccion_modelos[[
    "producto",
    "modelo_seleccionado",
    "RMSE_validacion",
    "MAE_validacion",
    "R2_validacion",
    "RMSE_prueba",
    "MAE_prueba",
    "R2_prueba"
]])

print("\nMétricas finales en prueba:")
display(df_metricas_finales)


Modelado con selección automática finalizado correctamente.
Reporte: /content/drive/MyDrive/resultados_modelado_mejor_modelo.xlsx
Modelos: /content/drive/MyDrive/modelos_mejor_modelo
Configuración: /content/drive/MyDrive/modelos_mejor_modelo/config_entrenamiento.pkl

Modelos seleccionados por producto:


,producto,modelo_seleccionado,RMSE_validacion,MAE_validacion,R2_validacion,RMSE_prueba,MAE_prueba,R2_prueba
0,almuerzo,HistGradientBoosting,14.892,11.943,0.1637,19.085,14.137,-0.1420
1,sopa,ElasticNet,25.222,16.307,0.0752,27.259,18.386,0.0217
2,fanesca,HistGradientBoosting,2.644,0.850,0.7471,0.062,0.032,0.0000
3,colada_morada,RandomForest,0.000,0.000,1.0000,6.148,2.554,0.7570



Métricas finales en prueba:


,producto,modelo_final,conjunto,registros,total_real,total_predicho,MAE,RMSE,R2,WAPE_%,MAPE_%,Sesgo_promedio
0,almuerzo,HistGradientBoosting,Prueba,111,11670.0,11862.311,14.137,19.085,-0.1420,13.447,12.805,1.733
1,sopa,ElasticNet,Prueba,111,7269.0,7572.864,18.386,27.259,0.0217,28.076,18.505,2.738
2,fanesca,HistGradientBoosting,Prueba,111,0.0,3.522,0.032,0.062,0.0000,NaN,NaN,0.032
3,colada_morada,RandomForest,Prueba,111,658.0,412.334,2.554,6.148,0.7570,43.083,39.087,-2.213
